### Scrape Jobs

In [ ]:
from pathlib import Path
from jobspy import scrape_jobs
import pandas as pd
from IPython.display import HTML, display

# https://github.com/speedyapply/JobSpy
jobs = scrape_jobs(
    site_name=["indeed", "linkedin"], # zip_recruiter, glassdoor, google, bayt, bdjobs
    linkedin_fetch_description=True,
    search_term='"AI Engineer"',
    location="Austria",
    country_indeed="Austria",
    job_type="fulltime",
    hours_old=240,
    results_wanted=10, 
)

df = pd.DataFrame(jobs)
print(f"Total jobs scraped: {len(df)}")
print(df[['title', 'company', 'site']]) 

### Filter

In [ ]:
# We ensure the title contains BOTH "AI" and "Engineer"
# We need to do this, because for indeed, also the job descriptions are searched, 
# so we could get job that only have "AI Engineer" in the description, but not in the title.
mask = (
    df['title'].str.contains('AI', case=False, na=False)
    & df['title'].str.contains('Engineer', case=False, na=False)
)
matching_jobs = df[mask].copy()

# We drop duplicates based on the combination of 'title' and 'company'
filtered_jobs = matching_jobs.drop_duplicates(subset=['title', 'company']).copy()

print(f"Total jobs found: {len(df)}")
print(f"Jobs matching 'AI' + 'Engineer' in title: {len(matching_jobs)}")
print(f"Jobs after deduplicating identical title/company pairs: {len(filtered_jobs)}")

### Save the jobs as JSONL file

In [ ]:
# Save the filtered jobs to a JSONL file in the "jobs" directory
# https://jsonlines.org/
output_dir = Path("jobs")
jsonl_path = output_dir / "1-scraped_jobs.jsonl"
filtered_jobs.to_json(jsonl_path, orient='records', lines=True, force_ascii=False)

### Display jobs

In [ ]:
results_to_show = filtered_jobs[['title', 'company', 'job_url']].copy()
results_to_show['job_url'] = results_to_show['job_url'].apply(
    lambda url: f'<a href=\"{url}\" target=\"_blank\">{url}</a>' if pd.notna(url) else ''
)

display(HTML(results_to_show.to_html(escape=False, index=False)))